[← Sumário da disciplina](README.md)

# Sensores Ativos — LiDAR II — pré-processamento e descritores de forma

**Maj Diego - 1° Semestre / 2027**

**Objetivos**

1. Analisar vizinhanças e remover pontos espúrios com SOR
2. Explicar CSF e classificação de terreno
3. Aplicar voxelização e downsampling com escala explícita
4. Derivar descritores geométricos locais

**Referências da aula**

- Open3D, *Point cloud outlier removal*.
- Zhang et al. (2016), *An Easy-to-Use Airborne LiDAR Data Filtering Method Based on Cloth Simulation*, Remote Sensing, 8(6), 501 (DOI: 10.3390/rs8060501).
- Weinmann et al. (2017), *Geometric features and their relevance for 3D point cloud classification* (DOI: 10.5194/isprs-annals-IV-1-W1-157-2017).

## O problema

Precisamos extrair o terreno e reconhecer estruturas de uma nuvem com densidade irregular e pontos espúrios. Como limpar e reduzir os dados preservando os elementos relevantes?

$$
\boxed{\text{Nuvem bruta}} \rightarrow \boxed{\text{Limpeza e amostragem}} \rightarrow \boxed{\text{Terreno e atributos}}
$$

## 1. Analisar vizinhanças e remover pontos espúrios com SOR

### 1.1. Suporte local
Uma vizinhança por $k$ vizinhos mantém aproximadamente o número de amostras, mas muda sua extensão conforme a densidade. Uma vizinhança por raio mantém a escala física, mas pode ter poucos pontos. A escolha deve acompanhar o tamanho das estruturas.

### 1.2. Statistical Outlier Removal
O SOR calcula, para cada ponto, a distância média aos vizinhos. Pontos com médias muito altas em relação à distribuição global podem ser rejeitados, conforme o número de vizinhos e um multiplicador do desvio padrão. O método pressupõe que isolamento espacial seja um indício útil de anomalia.

### 1.3. Risco de rejeição indevida
Fios, bordas e regiões pouco amostradas podem parecer isolados. Inspecionar os pontos rejeitados é tão necessário quanto inspecionar a nuvem limpa. Outliers não são sinônimos de todos os pontos fora do solo.

**Referência deste tópico:** Open3D, *Point cloud outlier removal*.

**Verificação de aprendizagem:** Que objetos válidos o SOR pode remover?

## 2. Explicar CSF e classificação de terreno

### 2.1. Simulação de tecido
No Cloth Simulation Filtering, a nuvem é invertida e um tecido simulado aproxima uma superfície de referência. A proximidade dos pontos à superfície auxilia a separar terreno e não terreno. A operação depende da geometria, não do número do retorno isoladamente.

### 2.2. Parâmetros e comportamento
Resolução do tecido, rigidez e limiar de classificação controlam a adaptação ao relevo. Tecido muito rígido pode ignorar detalhes topográficos; tecido muito flexível pode acompanhar estruturas indesejadas. Encostas, bordas e áreas sem amostras de solo exigem inspeção.

### 2.3. Produto intermediário
A classe de terreno obtida precisa ser avaliada antes de interpolar o MDT. Um telhado classificado como solo cria um erro sistemático na superfície e em todas as alturas normalizadas.

**Referência deste tópico:** Zhang et al. (2016), *An Easy-to-Use Airborne LiDAR Data Filtering Method Based on Cloth Simulation*, Remote Sensing, 8(6), 501 (DOI: 10.3390/rs8060501).

**Verificação de aprendizagem:** Explique como um erro na classe terreno afeta a altura normalizada.

## 3. Aplicar voxelização e downsampling com escala explícita

### 3.1. Partição do espaço
Para origem $\mathbf o$ e lado de voxel $s$, o índice é $\mathbf v=\lfloor(\mathbf p-\mathbf o)/s\rfloor$, por componente. Cada célula pode ser representada por um centroide, um ponto observado ou estatísticas. São produtos diferentes: o centroide não é necessariamente uma medição original.

### 3.2. Ganho e perda
Reduzir pontos acelera vizinhanças e registro, mas pode eliminar objetos menores que o voxel. Um voxel de 1 m não é adequado para preservar fios centimétricos. A origem da grade também influencia quais pontos são agrupados.

### 3.3. Preservar atributos
Rótulos podem usar voto majoritário; números de retorno e identificadores não devem ser simplesmente promediados como grandezas físicas. Para validação, guardar relação com a nuvem original e documentar a regra usada.

**Referência deste tópico:** Open3D, *Point cloud outlier removal*.

**Verificação de aprendizagem:** Qual atributo não deve ser promediado ao formar um voxel?

## 4. Derivar descritores geométricos locais

### 4.1. Covariância e autovalores
Com centroide $\bar{\mathbf p}$ de uma vizinhança, $\mathbf C=n^{-1}\sum(\mathbf p_i-\bar{\mathbf p})(\mathbf p_i-\bar{\mathbf p})^T$. Ordene $\lambda_1\ge\lambda_2\ge\lambda_3\ge0$. Os autovalores medem a dispersão nas direções principais.

### 4.2. Forma e normal
Para $\lambda_1>0$, linearidade $L=(\lambda_1-\lambda_2)/\lambda_1$, planaridade $P=(\lambda_2-\lambda_3)/\lambda_1$ e espalhamento $S=\lambda_3/\lambda_1$. Nessa definição, $L+P+S=1$. Uma vizinhança aproximadamente planar tem normal na direção do autovetor de menor autovalor, cujo sinal permanece ambíguo.

### 4.3. Interpretação e degeneração
Uma linha ideal tem $\lambda_1>0$ e os demais próximos de zero; um plano possui duas direções dominantes. Coincidência de pontos gera denominadores nulos: marcar o descritor como inválido é preferível a inventar estrutura. Mudar a vizinhança muda a geometria observada.

**Referência deste tópico:** Weinmann et al. (2017), *Geometric features and their relevance for 3D point cloud classification* (DOI: 10.5194/isprs-annals-IV-1-W1-157-2017).

**Verificação de aprendizagem:** Calcule L, P e S para autovalores 9, 3 e 1.